# 04 — Ablation Study (Experiments A4, A6, A7, A8)

**ICS555 Capstone: African Landmark Recognition**

This notebook runs and analyses all ablation experiments:

| Run | Model | Strategy | Key variable | Ablation question |
|-----|-------|----------|-------------|-------------------|
| A4  | ResNet-50 | full fine-tune | full aug | **Primary result** |
| A6  | ViT-B/16  | head-only      | full aug | Architecture: CNN vs Transformer |
| A7  | ResNet-50 | full fine-tune | minimal aug | Does RandAugment help when colour is a cue? |
| A8  | ResNet-50 | full + WRS     | full aug | Does fixing the sampler bug improve minority classes? |

In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !git clone https://github.com/zamsi-ajegetina/landmark-recognition.git
    %cd landmark-recognition
    !pip install -q -r requirements.txt
    from google.colab import drive
    drive.mount('/content/drive')
    !ln -sf /content/drive/MyDrive/landmark_images landmark_images

    import wandb
    wandb.login()

In [ ]:
%matplotlib inline
import sys, torch
sys.path.insert(0, '..')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Run A4 — ResNet-50 full fine-tune (primary result)

~60 min on T4. The main result we report in the paper.

In [ ]:
!python ablation_runner.py --config configs/A4_resnet50_full.yaml

## 2. Run A6 — ViT-B/16 head-only

Launch this overnight — ~10 hrs on T4 for 15 epochs.
Head-only (only 7,730 params trainable) to stay within Colab time limits.

In [ ]:
!python ablation_runner.py --config configs/A6_vitb16_frozen.yaml

## 3. Run A7 — ResNet-50 minimal augmentation

Same as A4 but with only horizontal flip (no RandAugment, no ColorJitter).
Tests: _does heavy colour augmentation hurt when the landmark colour/texture is discriminative?_

In [ ]:
!python ablation_runner.py --config configs/A7_resnet50_minimal_aug.yaml

## 4. Run A8 — ResNet-50 with WeightedRandomSampler

Same as A4 but with `use_weighted_sampler=True`.
Tests: _does correcting the class-imbalance sampler improve minority-class recall (macro-F1)?_

In [ ]:
!python ablation_runner.py --config configs/A8_resnet50_weighted.yaml

## 5. Results table

In [ ]:
import torch
from pathlib import Path
from src.data import get_data_loaders
from src.transfer import get_model_transfer_learning
from src.model import MyModel
from src.optimization import get_loss
from src.train import one_epoch_test
from sklearn.metrics import f1_score

data_loaders = get_data_loaders(batch_size=64, num_workers=2)
loss_fn = get_loss()

def eval_run(ckpt, model_fn):
    if not Path(ckpt).exists():
        return None, None, None
    m = model_fn()
    m.load_state_dict(torch.load(ckpt, map_location='cpu'))
    _, top1, top5, targets, preds = one_epoch_test(data_loaders['test'], m, loss_fn)
    mf1 = f1_score(targets, preds, average='macro', zero_division=0)
    return top1, top5, mf1

runs = [
    ('A0 scratch',       'checkpoints/A0_scratch.pt',          lambda: MyModel(50)),
    ('A1 RN18 frozen',   'checkpoints/A1_resnet18_frozen.pt',  lambda: get_model_transfer_learning('resnet18', 50, 'frozen')),
    ('A2 RN18 full',     'checkpoints/A2_resnet18_full.pt',    lambda: get_model_transfer_learning('resnet18', 50, 'full')),
    ('A4 RN50 full',     'checkpoints/A4_resnet50_full.pt',    lambda: get_model_transfer_learning('resnet50', 50, 'full')),
    ('A6 ViT-B/16',      'checkpoints/A6_vitb16_frozen.pt',    lambda: get_model_transfer_learning('vit_b_16', 50, 'frozen')),
    ('A7 RN50 min aug',  'checkpoints/A7_resnet50_minimal_aug.pt', lambda: get_model_transfer_learning('resnet50', 50, 'full')),
    ('A8 RN50 WRS',      'checkpoints/A8_resnet50_weighted.pt',    lambda: get_model_transfer_learning('resnet50', 50, 'full')),
]

print(f'{"Run":<22} {"Top-1":>7} {"Top-5":>7} {"Macro-F1":>10}')
print('─' * 50)
for name, ckpt, model_fn in runs:
    top1, top5, mf1 = eval_run(ckpt, model_fn)
    if top1 is None:
        print(f'{name:<22} (checkpoint not found)')
    else:
        print(f'{name:<22} {100*top1:6.1f}% {100*top5:6.1f}% {100*mf1:9.1f}%')

## 6. Ablation analysis plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Replace with real numbers after training
results = {
    'A0\nScratch': (56.0, 80.0, 38.0),
    'A1\nRN18 frozen': (72.0, 90.0, 62.0),
    'A2\nRN18 full': (75.0, 92.0, 65.0),
    'A4\nRN50 full': (None, None, None),  # fill after training
    'A6\nViT-B/16': (None, None, None),
    'A7\nMin aug': (None, None, None),
    'A8\nWRS': (None, None, None),
}

labels = [k for k, v in results.items() if v[0] is not None]
top1   = [results[k][0] for k in labels]
top5   = [results[k][1] for k in labels]
mf1    = [results[k][2] for k in labels]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, top1, width, label='Top-1 %', color='steelblue')
ax.bar(x,         top5, width, label='Top-5 %', color='seagreen')
ax.bar(x + width, mf1,  width, label='Macro-F1 %', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy / F1 (%)')
ax.set_title('Ablation results — African Landmark Recognition')
ax.legend()
ax.axhline(y=60, color='gray', linestyle='--', linewidth=0.8, label='Passing threshold')
plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=150)
plt.show()
print('Figure saved as ablation_comparison.png')